# Redes de personagens em Cyberpunk: Edgerunners (temporada 1)
45RPE, Resolução de Problemas com Estruturas de Dados, trabalho de recuperação.

## 1. Identificação da equipe
Trabalho feito individualmente por **Alison Natan Caetano**, Engenharia de Software, UDESC CEAVI (Ibirama). Disciplina 45RPE, prof. Marcelo de Souza.

**Como executar** deixe o arquivo `base_interacoes.txt` na mesma pasta deste notebook e rode todas as células em ordem. Bibliotecas necessárias: `pip install networkx pandas matplotlib scipy`. As figuras são salvas na pasta `graficos/` e os resultados em texto na pasta `resultados/`, as duas criadas automaticamente.

## 2. Introdução e objetivos
Meu objetivo é entender como as mortes da história mudam a rede de personagens de Cyberpunk: Edgerunners.

Para isso respondo três perguntas.

| Pergunta | Análises |
|---|---|
| 1. Quem sustenta a rede e o que se quebra quando cada personagem morre? | 3, 4, 5 e 13 |
| 2. Como a equipe de mercenários se forma e se desfaz ao longo dos episódios? | 7, 8 e 12 |
| 3. As traições do final aparecem na rede? | 6, 9 e 11 |

As análises 1, 2 e 10 descrevem a rede e dão o contexto para as outras.

## 3. Descrição da obra e delimitação do corpus
A obra é a primeira temporada de Cyberpunk: Edgerunners, disponível na Netflix, com 10 episódios. O corpus são os 10 episódios completos.

Escolhi a obra porque a história é uma sequência de perdas, e isso permite ver na rede o que cada morte quebra.

**Investigação de originalidade** busquei em 15/11/2026, no buscador web e no arXiv, os termos "Cyberpunk: Edgerunners" character network analysis, "Edgerunners" social network analysis graph theory e "Cyberpunk Edgerunners" scholarly article. A série já foi estudada por outras perspectivas, como uma análise pós-humanista, uma leitura psicanalítica do David e um trabalho sobre narrativa transmídia entre o jogo e o anime. Existem estudos de redes de personagens em outros animes e mangás (arXiv 2303.07208 e arXiv 2411.03333), mas não encontrei nenhum que analise Cyberpunk: Edgerunners por redes de interação entre personagens. A busca se baseou nos títulos e resumos dos resultados.

## 4. Definição do esquema de dados
**Registro** `personagem1 personagem2 sinal ambiente episodio minuto`, com os campos separados por um espaço em branco. A rede é não direcionada, então a ordem dos dois personagens não tem significado.

**Interação** conversa, ação dirigida ao outro (incluindo conflitos) ou ação conjunta em que os dois agem por um mesmo objetivo. Chamada em que os dois se respondem conta como interação direta. Interação indireta não entra.

**Início e fim** a cada mudança de cena ou de ambiente. Quando um dos personagens sai da cena, a interação termina.

**Eventos** um registro por cena, mesmo com várias falas. Cenas diferentes do mesmo par geram registros separados. Se a mesma cena muda de sinal, são dois registros com minutos diferentes.

**Sinal**, pelo resultado da interação para a equipe do David
- `+` a equipe ganha ou vence o confronto.
- `-` a equipe perde ou sai prejudicada.
- `0` interação direta sem ganho nem perda clara.
- Sem ninguém da equipe na cena, vale o resultado da ação (agressão é `-`, ajuda é `+`).

**Identificadores** `NomeSobrenome`, sem acento, espaço ou símbolo. Personagem sem nome recebe um ID descritivo, como `DiretorAcademia`, e um grupo sem nome que age junto vira um único ID, como `Tiras`. O dicionário com ID, nome canônico, apelidos e primeira aparição está no cabeçalho de `base_interacoes.txt`.

## 5. Descrição da coleta e validação das interações
Assisti à série episódio por episódio e anotei cada cena em áudios curtos, que depois viraram os registros. Para conferir afiliações e nomes usei a wiki da franquia (cyberpunk.fandom.com), por exemplo para confirmar que o Maxim e o Sr. Tanaka são da Arasaka e que o contato do Faraday no episódio 8 é da Arasaka.

A célula abaixo lê a base e valida cada registro: 6 campos, personagens presentes no dicionário, sinal em `+`, `-` ou `0`, e formato do episódio e do minuto. Se algum registro estiver errado, a execução para e mostra a linha.

In [ ]:
"""Código comum a todas as análises: leitura e validação da base, montagem das redes,
grupos dos personagens e estilo dos gráficos. Rode os scripts a partir da raiz do projeto."""
import os
import re
import sys
from collections import Counter

import matplotlib
if "ipykernel" not in sys.modules:      # fora do notebook, só grava as figuras em arquivo
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

BASE = "base_interacoes.txt"
os.makedirs("graficos", exist_ok=True)
os.makedirs("resultados", exist_ok=True)


# ---------- 1. leitura e validação da base ----------
def ler_base(caminho=BASE):
    """Devolve o dicionário de personagens e um DataFrame com um registro por linha."""
    personagens, linhas, erros = {}, [], []
    for n, linha in enumerate(open(caminho, encoding="utf-8").read().splitlines(), start=1):
        linha = linha.strip()
        if linha.startswith("# PERSONAGEM "):
            id_, nome_canonico, apelidos, primeira = [c.strip() for c in linha[13:].split("|")]
            personagens[id_] = {"nome_canonico": nome_canonico, "primeira_aparicao": primeira,
                                "apelidos": [a.strip() for a in apelidos.split(";") if a.strip()]}
            continue
        if not linha or linha.startswith("#"):
            continue
        campos = linha.split(" ")
        if len(campos) != 6:
            erros.append((n, f"esperados 6 campos, encontrados {len(campos)}"))
            continue
        p1, p2, sinal, ambiente, episodio, minuto = campos
        for p in (p1, p2):
            if p not in personagens:
                erros.append((n, f"ID fora do dicionário: {p}"))
        if p1 == p2:
            erros.append((n, "personagem1 e personagem2 são iguais"))
        if sinal not in ("+", "-", "0"):
            erros.append((n, f"sinal inválido: {sinal}"))
        if not re.fullmatch(r"[A-Za-z0-9]+", ambiente):
            erros.append((n, f"ambiente inválido: {ambiente}"))
        if not re.fullmatch(r"E\d{2}", episodio):
            erros.append((n, f"episódio inválido: {episodio}"))
        if not re.fullmatch(r"\d{2}:\d{2}", minuto):
            erros.append((n, f"minuto inválido: {minuto}"))
        linhas.append({"personagem1": p1, "personagem2": p2, "sinal": sinal, "ambiente": ambiente,
                       "episodio": episodio, "minuto": minuto})
    assert not erros, erros
    return personagens, pd.DataFrame(linhas)


personagens, df = ler_base()
EPISODIOS = sorted(df["episodio"].unique())


# ---------- 2. relatório de resultados ----------
class Relatorio:
    """Mostra o resultado na tela e grava o mesmo texto em resultados/."""

    def __init__(self, arquivo, titulo):
        self.caminho = f"resultados/{arquivo}.txt"
        self.linhas = []
        self.escrever(titulo)

    def escrever(self, texto=""):
        print(texto)
        self.linhas.append(str(texto))

    def tabela(self, tabela):
        self.escrever(tabela.to_string())

    def salvar(self):
        with open(self.caminho, "w", encoding="utf-8") as f:
            f.write("\n".join(self.linhas) + "\n")



print(f"{len(personagens)} personagens no dicionário e {len(df)} registros válidos")

## 6. Estatísticas descritivas da base

In [ ]:
rel = Relatorio("estatisticas_da_base", "ESTATÍSTICAS DESCRITIVAS DA BASE")
participam = set(df["personagem1"]) | set(df["personagem2"])
rel.escrever(f"registros: {len(df)}")
rel.escrever(f"personagens no dicionário: {len(personagens)}; com ao menos um registro: {len(participam)}")
rel.escrever("\nregistros por sinal:")
rel.tabela(df["sinal"].value_counts())
rel.escrever("\nregistros por episódio e sinal:")
rel.tabela(df.groupby(["episodio", "sinal"]).size().unstack(fill_value=0))
rel.escrever("\nregistros por ambiente:")
rel.tabela(df["ambiente"].value_counts())
rel.salvar()

## 7. Modelagem das redes
Cada vértice é um personagem e cada aresta liga dois personagens que interagiram. O peso da aresta é o número de registros do par, ou seja, em quantas cenas eles interagiram. A rede é não direcionada, porque as perguntas tratam de quem se liga a quem, e não de quem age sobre quem. A limitação é que a rede não mostra o sentido da ação.

O sinal entra de dois jeitos. Nas redes separadas, uma só com as interações positivas e outra só com as negativas. E no sinal líquido do par, que é a soma dos sinais das cenas dele: soma positiva dá par positivo, negativa dá par negativo e soma zero dá par neutro.

Também monto uma rede por episódio e uma rede acumulada, do episódio 1 até o episódio N.

Nos desenhos, a cor do vértice mostra o grupo do personagem (mercenários, Arasaka, Militech, fixers, polícia e outros), e cada grupo fica sempre na mesma região da figura.

**Dados ausentes ou ambíguos** nenhum registro ficou com sinal incerto. Três cenas sem local identificável usam o ambiente `LocalNaoIdentificado`.

In [ ]:
# ---------- 3. redes ----------
def montar_grafo(registros):
    """Grafo não direcionado. O peso da aresta é o número de registros (cenas) do par."""
    g = nx.Graph()
    for a, b in zip(registros["personagem1"], registros["personagem2"]):
        if g.has_edge(a, b):
            g[a][b]["weight"] += 1
        else:
            g.add_edge(a, b, weight=1)
    return g


G = montar_grafo(df)                              # rede global
G_pos = montar_grafo(df[df["sinal"] == "+"])      # só interações positivas
G_neg = montar_grafo(df[df["sinal"] == "-"])      # só interações negativas
G_ep = {ep: montar_grafo(df[df["episodio"] == ep]) for ep in EPISODIOS}      # uma rede por episódio
G_acum = {ep: montar_grafo(df[df["episodio"] <= ep]) for ep in EPISODIOS}    # do episódio 1 até o N


def sinal_por_par(registros):
    """Sinal líquido de cada par: soma das cenas (+ vale 1, - vale -1, 0 vale 0).
    Soma positiva dá '+', negativa dá '-' e soma zero dá '0' (neutro)."""
    soma = Counter()
    for a, b, s in zip(registros["personagem1"], registros["personagem2"], registros["sinal"]):
        soma[tuple(sorted((a, b)))] += {"+": 1, "-": -1, "0": 0}[s]
    return {par: "+" if v > 0 else "-" if v < 0 else "0" for par, v in soma.items()}


# ---------- 4. grupos e nomes ----------
EQUIPE = {"Maine", "LucynaKushinada", "Kiwi", "Dorio", "Pilar", "Rebecca", "Falco", "Julio"}
GRUPOS = {
    "Mercenários": EQUIPE | {"DavidMartinez"},
    "Arasaka": {"DiretorAcademia", "SrTanaka", "Katsuo", "SegurancaArasaka", "AgentesArasaka",
                "OficialArasaka", "AdamSmasher", "Maxim"},
    "Militech": {"Militech"},
    "Fixers": {"Faraday", "Wakako"},
    "Polícia": {"Tiras", "MaxTac"},
}
COR_GRUPO = {"Mercenários": "#2a78d6", "Arasaka": "#222222", "Militech": "#f08c00",
             "Fixers": "#8e44ad", "Polícia": "#1baf7a", "Outros": "#c9c9c4"}


def grupo(v):
    for g, membros in GRUPOS.items():
        if v in membros:
            return g
    return "Outros"


NOME_CURTO = {"DavidMartinez": "David", "LucynaKushinada": "Lucy", "GloriaMartinez": "Glória",
              "CyberpsicopataEp1": "Cyberpsicopata (ep. 1)", "CyberpsicopataEp4": "Cyberpsicopata (ep. 4)",
              "MaxTac": "MAX-TAC", "SrTanaka": "Sr. Tanaka", "DiretorAcademia": "Diretor",
              "TraumaTeam": "Trauma Team", "MedicoHospital": "Médico", "JK": "JK",
              "SegurancaArasaka": "Segurança Arasaka", "InimigosAssalto": "Inimigos (assalto)",
              "InimigosGangue": "Inimigos (gangue)", "InimigosCampoAberto": "Inimigos (campo)",
              "AgentesArasaka": "Agentes Arasaka", "OficialArasaka": "Oficial Arasaka",
              "AdamSmasher": "Adam Smasher", "DronesArmados": "Drones", "ChefeGangue": "Chefe da gangue"}


def nome(v):
    return NOME_CURTO.get(v, re.sub(r"(?<!^)(?=[A-Z])", " ", v))


# ---------- 5. estilo dos gráficos ----------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e6e3",
    "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
COR_SINAL = {"+": "#1a9850", "-": "#d62f2f", "0": "#7a7a7a"}
ESTILO_SINAL = {"+": "solid", "-": "dashed", "0": "dotted"}     # estilo repete a informação da cor
NOME_SINAL = {"+": "positiva", "-": "negativa", "0": "neutra"}
COR_BARRA = "#2a78d6"
COR_COMUNIDADE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#52514e"]


def salvar(fig, arquivo):
    """Grava a figura em graficos/ e mostra no notebook."""
    fig.savefig(f"graficos/{arquivo}.png", bbox_inches="tight", pad_inches=0.15, facecolor="white")
    if "ipykernel" in sys.modules:
        plt.show()
    plt.close(fig)


# ---------- 6. posição dos personagens no desenho da rede ----------
# Cada grupo ocupa uma região fixa, a mesma em todas as figuras: mercenários no centro, com o David
# no meio, Arasaka à direita, polícia em cima, fixers e Militech embaixo, e os outros num arco à esquerda.
ORDEM_MERCENARIOS = ["Maine", "Dorio", "Kiwi", "LucynaKushinada", "Pilar", "Rebecca", "Julio", "Falco"]
CENTRO = {"Arasaka": (1.5, 0.0), "Militech": (1.9, -1.0), "Fixers": (1.05, -1.2), "Polícia": (0.2, 1.4)}
POSICAO_FIXA = {"CyberpsicopataEp1": (1.1, 1.4)}     # só se liga à polícia e à segurança da Arasaka


def _circulo(nos, centro, raio, inicio=90.0, abertura=360.0):
    passo = abertura / max(len(nos), 1)
    return {v: (centro[0] + raio * np.cos(np.radians(inicio + i * passo)),
                centro[1] + raio * np.sin(np.radians(inicio + i * passo))) for i, v in enumerate(nos)}


ORDEM_ANEL = {}


def calcular_posicoes(g):
    pos = {"DavidMartinez": (0.0, 0.0)}
    merc = [v for v in ORDEM_MERCENARIOS if v in g]
    pos.update(_circulo(merc, (0, 0), 0.42, inicio=100))
    # quem tem mais ligações fica do lado de dentro do grupo, virado para os mercenários
    for gr, centro in CENTRO.items():
        nos = sorted((v for v in g if grupo(v) == gr), key=lambda v: (v in ("SegurancaArasaka", "MaxTac"), -g.degree(v), v))
        if len(nos) <= 2:
            pos.update({v: (centro[0] + 0.28 * i, centro[1]) for i, v in enumerate(nos)})
        else:
            pos.update(_circulo(nos, centro, 0.36, inicio=180))
    # os outros ficam num arco à esquerda, ordenados pelo ângulo médio dos vizinhos mercenários
    pos.update({v: p for v, p in POSICAO_FIXA.items() if v in g})
    outros = [v for v in g if v not in pos]

    def angulo(v):
        viz = [pos[u] for u in g[v] if u in merc]
        if not viz:
            return 400.0
        a = np.degrees(np.arctan2(np.mean([p[1] for p in viz]), np.mean([p[0] for p in viz]))) % 360
        return a if a >= 60 else a + 360
    outros.sort(key=lambda v: (angulo(v), v))
    pos.update(_circulo(outros, (0, 0), 1.15, inicio=100, abertura=205))
    ORDEM_ANEL.update({v: i for i, v in enumerate(outros)})
    return {v: np.array(p) for v, p in pos.items()}


POS = calcular_posicoes(G)
_PESO_MAX = max(dict(G.degree(weight="weight")).values())


def tamanho(g, v):
    """Tamanho do vértice pela raiz do número de cenas do personagem, numa faixa estreita."""
    return 130 + 520 * (g.degree(v, weight="weight") / _PESO_MAX) ** 0.5


def _posicao_rotulo(v):
    """O nome fica do lado de fora do grupo do personagem, para não cair em cima das arestas."""
    x, y = POS[v]
    if v == "DavidMartinez":
        return x, y - 0.12, "center", "top"
    gr = grupo(v)
    cx, cy = (0.0, 0.0) if gr in ("Mercenários", "Outros") else CENTRO[gr]
    if gr in ("Militech", "Fixers", "Polícia") or v in POSICAO_FIXA:
        return x, y + 0.09, "center", "bottom"
    d = np.array([x - cx, y - cy])
    d = d / (np.linalg.norm(d) or 1)
    ha = "left" if d[0] > 0.3 else "right" if d[0] < -0.3 else "center"
    va = "bottom" if d[1] > 0.3 else "top" if d[1] < -0.3 else "center"
    # no arco, nomes vizinhos que ficam acima ou abaixo do vértice alternam a distância para não encostar
    dist = 0.17 if (ha == "center" and ORDEM_ANEL.get(v, 0) % 2) else 0.075
    return x + dist * d[0], y + dist * d[1], ha, va


def desenhar_rede(ax, g, cor_vertice=None, sinais=None, largura_peso=False):
    """Desenha a rede g. cor_vertice: dict vértice -> cor (padrão: cor do grupo).
    sinais: dict par -> '+', '-' ou '0', que define cor e estilo das arestas."""
    cor_vertice = cor_vertice or {v: COR_GRUPO[grupo(v)] for v in g}
    if sinais:
        for s in ["0", "+", "-"]:
            arestas = [e for e in g.edges() if sinais.get(tuple(sorted(e))) == s]
            nx.draw_networkx_edges(g, POS, edgelist=arestas, ax=ax, edge_color=COR_SINAL[s],
                                   style=ESTILO_SINAL[s], width=1.3, alpha=0.75)
    else:
        larguras = [0.6 + 0.3 * g[u][v]["weight"] for u, v in g.edges()] if largura_peso else 0.9
        nx.draw_networkx_edges(g, POS, ax=ax, edge_color="#a7a69f", width=larguras, alpha=0.6)
    nx.draw_networkx_nodes(g, POS, ax=ax, node_color=[cor_vertice[v] for v in g],
                           node_size=[tamanho(G, v) for v in g], edgecolors="white", linewidths=1.3)
    for v in g:
        x, y, ha, va = _posicao_rotulo(v)
        ax.text(x, y, nome(v), ha=ha, va=va, fontsize=8.5, color="#0b0b0b", zorder=5,
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.8))
    ax.set_xlim(-1.75, 2.35)
    ax.set_ylim(-1.5, 1.65)
    ax.set_aspect("equal")
    ax.axis("off")


def legenda_grupos(fig, g):
    usados = [k for k in COR_GRUPO if any(grupo(v) == k for v in g)]
    fig.legend(handles=[Line2D([0], [0], marker="o", color="none", markerfacecolor=COR_GRUPO[k],
                               markeredgecolor="white", markersize=10, label=k) for k in usados],
               title="Grupo (cor do vértice)", loc="lower left", bbox_to_anchor=(0.06, 0.0),
               fontsize=9, title_fontsize=9, ncol=3)


def legenda_sinais(fig, sinais=("+", "-", "0")):
    fig.legend(handles=[Line2D([0], [0], color=COR_SINAL[s], linestyle=ESTILO_SINAL[s], lw=2,
                               label=NOME_SINAL[s]) for s in sinais],
               title="Sinal líquido do par (aresta)", loc="lower right", bbox_to_anchor=(0.94, 0.0),
               fontsize=9, title_fontsize=9, ncol=3)


print(f"Python {sys.version.split()[0]}, NetworkX {nx.__version__}, pandas {pd.__version__}, Matplotlib {matplotlib.__version__}")
print(f"rede global: {G.number_of_nodes()} vértices e {G.number_of_edges()} arestas")
print(f"rede positiva: {G_pos.number_of_nodes()} vértices e {G_pos.number_of_edges()} arestas")
print(f"rede negativa: {G_neg.number_of_nodes()} vértices e {G_neg.number_of_edges()} arestas")

## 8. Análises

### Análise 1. Caracterização geral da rede
**Pergunta e objetivo** qual o tamanho da rede e quão ligados os personagens estão entre si.

**Técnica** número de vértices e arestas, densidade, grau médio, agrupamento médio, diâmetro e caminho médio.

**Configuração** rede global não direcionada. Densidade e agrupamento ignoram o peso. Diâmetro e caminho médio usam o maior componente, que aqui é a rede inteira. A figura mostra a rede com a cor do vértice pelo grupo e a aresta pelo sinal líquido do par.

**Resultados**

In [ ]:
rel = Relatorio("analise01", "ANÁLISE 1. Caracterização geral da rede")

# diâmetro e caminho médio só existem onde todos se alcançam, por isso uso o maior componente
maior = G.subgraph(max(nx.connected_components(G), key=len))
par = max(G.edges(data="weight"), key=lambda e: e[2])

rel.escrever(f"registros: {len(df)}")
rel.escrever(f"vértices: {G.number_of_nodes()}")
rel.escrever(f"arestas: {G.number_of_edges()}")
rel.escrever(f"par com mais cenas: {nome(par[0])} e {nome(par[1])} ({par[2]} cenas)")
rel.escrever(f"densidade: {nx.density(G):.3f}")
rel.escrever(f"grau médio: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")
rel.escrever(f"agrupamento médio: {nx.average_clustering(G):.3f}")
rel.escrever(f"componentes conexos: {nx.number_connected_components(G)}")
rel.escrever(f"diâmetro: {nx.diameter(maior)}")
rel.escrever(f"caminho médio: {nx.average_shortest_path_length(maior):.2f}")
rel.salvar()

# rede global: cor do vértice pelo grupo, cor e estilo da aresta pelo sinal líquido do par
fig, ax = plt.subplots(figsize=(13, 10))
desenhar_rede(ax, G, sinais=sinal_por_par(df))
legenda_grupos(fig, G)
legenda_sinais(fig)
ax.set_title("Rede global de interações, temporada 1\n"
             "(tamanho do vértice pela raiz do número de cenas do personagem)")
salvar(fig, "fig01_rede_global")

**Interpretação** A temporada fecha com 37 personagens e 84 pares que interagem. A densidade é 0,13, então só 13% dos pares possíveis chegam a interagir. Mesmo assim dois personagens quaisquer estão em média a 2,23 passos um do outro, e o caminho mais longo tem 6 passos. O agrupamento médio de 0,57 é bem maior que a densidade, o que mostra que os personagens formam grupos fechados em volta de alguns nomes.

**Conclusões**
- Rede de 37 personagens e 84 relações, toda conectada.
- Rede esparsa, mas com distâncias curtas, o que indica alguém no meio encurtando os caminhos.

### Análise 2. Personagens mais conectados
**Pergunta e objetivo** quem interage com mais personagens e quem aparece em mais cenas.

**Técnica** grau (número de personagens diferentes com quem interage) e grau ponderado (número de cenas de interação).

**Configuração** rede global. Gráfico de barras com os 12 maiores graus ponderados.

**Resultados**

In [ ]:
rel = Relatorio("analise02", "ANÁLISE 2. Personagens mais conectados")

# grau: com quantos personagens diferentes interage; grau ponderado: em quantas cenas aparece
graus = pd.DataFrame({
    "grau": dict(G.degree()),
    "grau_ponderado": dict(G.degree(weight="weight")),
}).sort_values("grau_ponderado", ascending=False)
graus.index = [nome(v) for v in graus.index]
rel.tabela(graus.head(12))
rel.salvar()

top = graus.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index, top["grau_ponderado"], color=COR_BARRA, height=0.6)
for y, v in enumerate(top["grau_ponderado"]):
    ax.text(v + 1, y, str(v), va="center", fontsize=8, color="#52514e")
ax.set_xlabel("grau ponderado (número de cenas de interação)")
ax.set_title("Os 12 personagens com maior grau ponderado")
ax.grid(axis="y", visible=False)
salvar(fig, "fig02_grau_ponderado")

**Interpretação** O David aparece em 116 cenas de interação e interage com 30 dos outros 36 personagens. Depois dele vêm a Lucy, com 42 cenas, e o Maine, com 40, e em seguida Rebecca e Kiwi, com 29 cada. O David tem quase três vezes o segundo lugar. Mesmo morrendo no episódio 6, o Maine termina como o terceiro mais conectado.

**Conclusões**
- O David concentra as interações da rede.
- Lucy e Maine formam o segundo nível, e Rebecca e Kiwi o terceiro.

### Análise 3. Centralidade de intermediação
**Pergunta e objetivo** quem funciona como ponte entre partes diferentes da rede.

**Técnica** betweenness centrality normalizada.

**Configuração** rede global sem peso, porque aqui o peso significa proximidade (mais cenas juntos) e o algoritmo trataria o peso como distância.

**Resultados**

In [ ]:
rel = Relatorio("analise03", "ANÁLISE 3. Centralidade de intermediação")

# sem peso: aqui o peso é número de cenas (proximidade) e o algoritmo trataria peso como distância
intermediacao = pd.Series(nx.betweenness_centrality(G, normalized=True)).sort_values(ascending=False)
intermediacao.index = [nome(v) for v in intermediacao.index]
rel.tabela(intermediacao.round(3).head(12))
rel.salvar()

top = intermediacao.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top.index, top.values, color=COR_BARRA, height=0.6)
for y, v in enumerate(top.values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
ax.set_xlabel("intermediação normalizada (betweenness)")
ax.set_title("Os 10 personagens com maior intermediação")
ax.grid(axis="y", visible=False)
salvar(fig, "fig03_intermediacao")

**Interpretação** O David tem intermediação de 0,67, então a maior parte dos caminhos mais curtos passa por ele. O Maine vem em segundo, com 0,21. Em terceiro aparece a MAX-TAC, com 0,11, porque é ela que liga o surto do episódio 1 ao resto da história quando enfrenta o Maine no episódio 6. O cyberpsicopata do episódio 1 e o Adam Smasher aparecem com 0,06 porque cada um é a única ligação de um personagem da Arasaka com o resto.

**Conclusões**
- O David é a ponte principal da rede e o Maine a segunda.
- A MAX-TAC liga o episódio 1 ao resto da história.

### Análise 4. Centralidade de proximidade
**Pergunta e objetivo** quem está mais perto de todos os outros personagens.

**Técnica** closeness centrality com a correção de Wasserman e Faust (`wf_improved=True`).

**Configuração** rede global sem peso. A correção reduz o valor de quem está em componente pequeno. Na temporada completa a rede é um componente só, mas até o episódio 5 o grupo do surto do episódio 1 ficava isolado, e mantive a correção para o cálculo valer nos dois casos.

**Resultados**

In [ ]:
rel = Relatorio("analise04", "ANÁLISE 4. Centralidade de proximidade")

# wf_improved=True é a correção de Wasserman e Faust, que reduz o valor de quem está em componente pequeno
proximidade = pd.Series(nx.closeness_centrality(G, wf_improved=True)).sort_values(ascending=False)
proximidade.index = [nome(v) for v in proximidade.index]
rel.tabela(proximidade.round(3))
rel.salvar()

top = pd.concat([proximidade.head(8), proximidade.tail(4)]).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(top.index, top.values, color=COR_BARRA, height=0.6)
for y, v in enumerate(top.values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
ax.axhline(3.5, color="#52514e", linestyle="--", lw=0.8)
ax.set_xlabel("proximidade (closeness)")
ax.set_title("Proximidade: os 8 maiores valores e os 4 menores")
ax.grid(axis="y", visible=False)
salvar(fig, "fig04_proximidade")

**Interpretação** O David tem proximidade de 0,80, bem acima do Maine, com 0,59, e da Lucy, com 0,56. Do David se chega a qualquer personagem em poucos passos. No fim da lista estão a segurança da Arasaka e o cyberpsicopata do episódio 1, que só se ligam ao resto pela MAX-TAC, e o oficial da Arasaka, que só aparece com o Adam Smasher.

**Conclusões**
- O David é o personagem mais próximo de todos.
- Os personagens do episódio 1 e o oficial da Arasaka ficam na periferia.

### Análise 5. Componentes conexos e o papel do David
**Pergunta e objetivo** existem grupos isolados, e o que acontece com a rede se o David sair dela.

**Técnica** componentes conexos e remoção de vértice. Técnica não listada no enunciado.

**Configuração** rede global e a mesma rede sem o vértice `DavidMartinez`. A figura mostra quem continua ligado e quem fica isolado.

**Resultados**

In [ ]:
rel = Relatorio("analise05", "ANÁLISE 5. Componentes conexos e o papel do David")


def componentes(grafo):
    return sorted((sorted(nome(v) for v in c) for c in nx.connected_components(grafo)), key=len, reverse=True)


rel.escrever(f"rede global: {nx.number_connected_components(G)} componente(s), o maior com "
             f"{len(componentes(G)[0])} personagens")

sem_david = G.copy()
sem_david.remove_node("DavidMartinez")
comps = componentes(sem_david)
rel.escrever(f"rede sem o David: {len(comps)} componentes, o maior com {len(comps[0])} personagens")
isolados = sorted(c[0] for c in comps if len(c) == 1)
rel.escrever(f"ficam isolados ({len(isolados)}): {', '.join(isolados)}")
rel.salvar()

# quem continua ligado ao grupo principal e quem fica isolado quando o David sai
principal = max(nx.connected_components(sem_david), key=len)
COR_ESTADO = {"continua ligado": "#2a78d6", "fica isolado": "#d62f2f"}
fig, ax = plt.subplots(figsize=(13, 10))
desenhar_rede(ax, sem_david, cor_vertice={v: COR_ESTADO["continua ligado" if v in principal else "fica isolado"]
                                         for v in sem_david})
fig.legend(handles=[Patch(color=c, label=k) for k, c in COR_ESTADO.items()], loc="lower left",
           bbox_to_anchor=(0.06, 0.0), fontsize=9, ncol=2)
ax.set_title("Rede global sem o David: quem continua ligado e quem fica isolado")
salvar(fig, "fig05_rede_sem_o_david")

**Interpretação** A rede da temporada é um componente só. Quando tiro o David ela quebra em 7 pedaços. Aldo, o chefe da gangue, Katsuo, o médico, o sequestrado e a Wakako ficam isolados, porque só se ligavam ao resto por ele. Os outros 30 personagens continuam ligados entre si, então a equipe e os inimigos dela se sustentam sem o David, mas a vida dele fora da equipe só existe através dele.

**Conclusões**
- Sem o David, 6 personagens ficam isolados e a rede quebra em 7 componentes.
- O núcleo da equipe e dos inimigos continua ligado sem ele.

### Análise 6. Detecção de comunidades
**Pergunta e objetivo** os grupos que a rede forma batem com os grupos da história, e onde cai o Faraday, que trai a equipe.

**Técnica** algoritmo de Louvain, que maximiza a modularidade.

**Configuração** rede global com peso, `seed=42` e resolução padrão. Como o Louvain tem uma parte aleatória, rodo 100 vezes com seeds diferentes e comparo com o algoritmo guloso de modularidade (Clauset, Newman e Moore), que não tem sorteio.

**Resultados**

In [ ]:
rel = Relatorio("analise06", "ANÁLISE 6. Detecção de comunidades")

comunidades = sorted(nx.community.louvain_communities(G, weight="weight", seed=42), key=len, reverse=True)
modularidade = nx.community.modularity(G, comunidades, weight="weight")
rel.escrever(f"modularidade (Louvain, seed 42): {modularidade:.3f}")
for i, c in enumerate(comunidades, start=1):
    rel.escrever(f"comunidade {i} ({len(c)}): {', '.join(sorted(nome(v) for v in c))}")

# o Louvain tem parte aleatória: rodo 100 vezes e conto quantas vezes cada par cai na mesma comunidade
pares_teste = [("Faraday", "Kiwi"), ("Faraday", "AgentesArasaka"), ("Maine", "Dorio"),
               ("DavidMartinez", "LucynaKushinada"), ("Kiwi", "Rebecca"), ("DavidMartinez", "Maine")]
juntos, mods = Counter(), []
for semente in range(100):
    com = nx.community.louvain_communities(G, weight="weight", seed=semente)
    mods.append(nx.community.modularity(G, com, weight="weight"))
    rotulo = {v: i for i, c in enumerate(com) for v in c}
    for a, b in pares_teste:
        juntos[(a, b)] += rotulo[a] == rotulo[b]
rel.escrever(f"\nmodularidade em 100 execuções: de {min(mods):.3f} a {max(mods):.3f}")
rel.escrever("vezes em que o par caiu na mesma comunidade, em 100 execuções:")
for a, b in pares_teste:
    rel.escrever(f"  {nome(a)} e {nome(b)}: {juntos[(a, b)]}")

# comparação com um algoritmo sem sorteio (guloso de Clauset, Newman e Moore)
guloso = nx.community.greedy_modularity_communities(G, weight="weight")
do_faraday = [c for c in guloso if "Faraday" in c][0]
rel.escrever(f"\ncomunidade do Faraday no algoritmo guloso: {', '.join(sorted(nome(v) for v in do_faraday))}")
rel.salvar()

cor = {v: COR_COMUNIDADE[i] for i, c in enumerate(comunidades) for v in c}
fig, ax = plt.subplots(figsize=(13, 10))
desenhar_rede(ax, G, cor_vertice=cor)
fig.legend(handles=[Patch(color=COR_COMUNIDADE[i], label=f"Comunidade {i + 1} ({len(c)} personagens)")
                    for i, c in enumerate(comunidades)],
           title="Comunidades (Louvain, seed 42)", loc="lower left", bbox_to_anchor=(0.06, 0.0),
           fontsize=9, title_fontsize=9, ncol=3)
ax.set_title(f"Comunidades detectadas pelo Louvain, modularidade {modularidade:.2f}")
salvar(fig, "fig06_comunidades")

**Interpretação** A modularidade ficou em 0,17, um valor baixo, porque o David liga quase todo mundo. Mesmo assim as comunidades batem com a história. Uma junta o Maine, a Dorio, o JK, o Sr. Tanaka, os tiras e o Trauma Team, que é o arco do Maine até a morte dele. Outra junta Kiwi, Rebecca, Falco, Pilar e Julio com os inimigos que a equipe enfrenta. O David fica com a Lucy e com os personagens que só falam com ele, e o surto do episódio 1 forma a própria comunidade.

O Faraday fica numa comunidade separada só com os agentes da Arasaka, fora da equipe. Isso não é sorte do algoritmo, porque o resultado se repete nas 100 execuções e também no algoritmo guloso. Com a Kiwi, que trai junto com ele, o Faraday só cai na mesma comunidade em 25 das 100 execuções.

**Conclusões**
- Modularidade baixa, por causa do peso do David.
- O Faraday fica com a Arasaka e fora da equipe em 100 de 100 execuções.
- Maine e Dorio, e David e Lucy, ficam juntos em todas as execuções.

### Análise 7. Evolução por episódio
**Pergunta e objetivo** como a rede cresce e muda ao longo dos episódios.

**Técnica** métricas por episódio (vértices, arestas, densidade, grau do David) e na rede acumulada.

**Configuração** uma rede por episódio e uma rede acumulada do episódio 1 até o N. Dois gráficos de linha e o desenho da rede de cada episódio, com os personagens sempre na mesma posição para dar para comparar.

**Resultados**

In [ ]:
rel = Relatorio("analise07", "ANÁLISE 7. Evolução por episódio")

linhas = []
for ep in EPISODIOS:
    g, acum = G_ep[ep], G_acum[ep]
    linhas.append({
        "episodio": ep,
        "registros": int((df["episodio"] == ep).sum()),
        "vertices_ep": g.number_of_nodes(),
        "arestas_ep": g.number_of_edges(),
        "densidade_ep": round(nx.density(g), 2),
        "grau_David_ep": g.degree("DavidMartinez") if "DavidMartinez" in g else 0,
        "vertices_acum": acum.number_of_nodes(),
        "densidade_acum": round(nx.density(acum), 2),
    })
evolucao = pd.DataFrame(linhas).set_index("episodio")
rel.tabela(evolucao)
rel.salvar()

rotulos = [f"E{int(e[1:])}" for e in EPISODIOS]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(rotulos, evolucao["densidade_ep"], marker="o", color=COR_BARRA, lw=2, markersize=6)
for x, y in zip(rotulos, evolucao["densidade_ep"]):
    ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("densidade da rede do episódio")
ax.set_title("Densidade da rede de cada episódio")
salvar(fig, "fig07a_densidade_por_episodio")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(rotulos, evolucao["vertices_acum"], marker="o", color=COR_BARRA, lw=2, markersize=6)
for x, y in zip(rotulos, evolucao["vertices_acum"]):
    ax.annotate(str(y), (x, y), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
ax.set_ylim(0, evolucao["vertices_acum"].max() * 1.15)
ax.set_ylabel("personagens na rede acumulada")
ax.set_title("Personagens acumulados até cada episódio")
salvar(fig, "fig07b_personagens_acumulados")

# uma figura por episódio, com os personagens na mesma posição da rede global, para comparar
for ep in EPISODIOS:
    g = G_ep[ep]
    fig, ax = plt.subplots(figsize=(13, 10))
    desenhar_rede(ax, g, sinais=sinal_por_par(df[df["episodio"] == ep]))
    legenda_grupos(fig, g)
    legenda_sinais(fig)
    ax.set_title(f"Rede do episódio {int(ep[1:])}: {g.number_of_nodes()} personagens e {g.number_of_edges()} pares")
    salvar(fig, f"fig07c_rede_episodio_{ep[1:]}")

**Interpretação** A densidade sobe do episódio 1 (0,16) até o 5 (0,75), que é o auge da equipe agindo junta no plano do JK. Ela cai no episódio 6, com a cyberpsicose do Maine, e volta a subir no 9 (0,62), quando a equipe que sobrou se junta para o último trabalho. O número de personagens acumulados cresce de 11 para 37, mas os personagens novos da segunda metade são quase todos inimigos ou figuras de um trabalho só. O grau do David tem o pico no episódio 7, com 11, quando ele assume a liderança.

**Conclusões**
- A rede fica mais fechada nos episódios em que a equipe age junta (5 e 9).
- O episódio 6 quebra a equipe.
- O David chega ao maior grau no episódio 7, já como líder.

### Análise 8. Comparação entre sinais
**Pergunta e objetivo** como se distribuem as interações positivas, negativas e neutras, por episódio e por personagem.

**Técnica** contagem de sinais por episódio e grau ponderado nas redes positiva e negativa.

**Configuração** sinal pelo resultado para a equipe do David (seção 4). Gráfico de barras empilhadas.

**Resultados**

In [ ]:
rel = Relatorio("analise08", "ANÁLISE 8. Comparação entre sinais")

rel.escrever("registros por sinal na temporada:")
rel.tabela(df["sinal"].value_counts())
sinais_ep = df.groupby(["episodio", "sinal"]).size().unstack(fill_value=0).reindex(columns=["+", "-", "0"], fill_value=0)
rel.escrever("\nregistros por episódio e sinal:")
rel.tabela(sinais_ep)

# grau ponderado de cada personagem na rede positiva e na negativa
por_personagem = pd.DataFrame({
    "positivas": dict(G_pos.degree(weight="weight")),
    "negativas": dict(G_neg.degree(weight="weight")),
}).fillna(0).astype(int).sort_values("negativas", ascending=False)
por_personagem.index = [nome(v) for v in por_personagem.index]
rel.escrever("\ninterações positivas e negativas por personagem (10 com mais negativas):")
rel.tabela(por_personagem.head(10))
rel.salvar()

rotulos = [f"E{int(e[1:])}" for e in sinais_ep.index]
fig, ax = plt.subplots(figsize=(8, 4.2))
base = None
for s in ["+", "-", "0"]:
    ax.bar(rotulos, sinais_ep[s], bottom=base, color=COR_SINAL[s], label=NOME_SINAL[s], width=0.62,
           edgecolor="white", linewidth=1.5)
    base = sinais_ep[s] if base is None else base + sinais_ep[s]
ax.set_ylabel("número de interações")
ax.set_title("Sinal das interações por episódio")
ax.legend(title="Sinal", fontsize=8, title_fontsize=8, loc="upper left")
ax.grid(axis="x", visible=False)
salvar(fig, "fig08_sinais_por_episodio")

**Interpretação** Na temporada são 122 interações positivas, 68 negativas e 15 neutras. Só dois episódios têm maioria negativa. O episódio 1, com 11 de 15, que é a morte da mãe do David, e o episódio 6, com 18 de 27, que é a cyberpsicose do Maine. Depois do David, que acumula 32 interações negativas, vêm o Maine (15), a Lucy (11) e a Dorio e o Faraday (8 cada). O Faraday tem 8 negativas e só 1 positiva.

**Conclusões**
- Os episódios 1 e 6 são os mais negativos, cada um com uma perda grande para o David.
- Quase todas as interações do Faraday são negativas.

### Análise 9. Viradas de sinal entre pares
**Pergunta e objetivo** quais pares mudam de sinal ao longo da história, que é como uma traição pode aparecer na rede.

**Técnica** para cada par, a sequência dos sinais em ordem de episódio e minuto. Um par com `+` e `-` na sequência tem virada. Técnica não listada no enunciado.

**Configuração** todos os registros, ordenados por episódio e minuto. Gráfico de linha do tempo com os pares que viram.

**Resultados**

In [ ]:
rel = Relatorio("analise09", "ANÁLISE 9. Viradas de sinal entre pares")

# para cada par, a sequência dos sinais em ordem de episódio e minuto
ordenado = df.sort_values(["episodio", "minuto"])
ordenado = ordenado.assign(par=[" e ".join(sorted([nome(a), nome(b)]))
                                for a, b in zip(ordenado["personagem1"], ordenado["personagem2"])])
sequencias = ordenado.groupby("par")["sinal"].apply(list)
viradas = sequencias[sequencias.apply(lambda s: "+" in s and "-" in s)]   # par com + e - tem virada
rel.escrever(f"pares com virada de sinal: {len(viradas)} de {len(sequencias)}\n")
for par, s in viradas.items():
    rel.escrever(f"{par:34s} {' '.join(s)}")
rel.salvar()

ordem_ep = {ep: i for i, ep in enumerate(EPISODIOS)}
MARCA = {"+": "o", "-": "s", "0": "D"}
fig, ax = plt.subplots(figsize=(10, 0.8 + 0.42 * len(viradas)))
for y, par in enumerate(viradas.index):
    sub = ordenado[ordenado["par"] == par]
    x = [ordem_ep[e] + int(m[:2]) / 25 for e, m in zip(sub["episodio"], sub["minuto"])]
    ax.plot(x, [y] * len(x), color="#d7d6d1", lw=1, zorder=1)
    for xi, s in zip(x, sub["sinal"]):
        ax.scatter(xi, y, color=COR_SINAL[s], marker=MARCA[s], s=34, zorder=2, edgecolors="white", linewidths=0.6)
ax.set_yticks(range(len(viradas)))
ax.set_yticklabels(viradas.index, fontsize=8)
ax.set_xticks(range(len(EPISODIOS)))
ax.set_xticklabels([f"E{int(e[1:])}" for e in EPISODIOS])
ax.set_xlabel("episódio (cada ponto é uma cena, na ordem do minuto)")
ax.grid(axis="y", visible=False)
ax.legend(handles=[Line2D([0], [0], marker=MARCA[s], color="w", markerfacecolor=COR_SINAL[s], markersize=7,
                          label=NOME_SINAL[s]) for s in ["+", "-", "0"]],
          title="Sinal", loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8, title_fontsize=8)
ax.set_title("Pares que mudam de sinal ao longo da temporada")
salvar(fig, "fig09_viradas_de_sinal")

**Interpretação** 19 dos 84 pares mudam de sinal. A traição da Kiwi aparece aqui. Ela é positiva com a Lucy até o episódio 8, quando ajuda a capturar a Lucy, e com o Falco ela vira negativa no episódio 9. Com o Faraday é diferente, porque ele nunca tem uma interação positiva com a equipe, e o único `+` dele é o David vencendo ele no episódio 10. O David e o Maine terminam em `-` por causa da cyberpsicose do Maine. Nem toda virada é traição, como o diretor da Academia, que expulsa o David e depois oferece a bolsa que ele recusa.

**Conclusões**
- A traição da Kiwi aparece como virada de sinal com a Lucy e com o Falco.
- O Faraday não vira, porque nunca colabora com a equipe.
- A cyberpsicose do Maine vira o sinal dele com o David e com a Dorio.

### Análise 10. Distribuição dos graus
**Pergunta e objetivo** os personagens têm conexões parecidas ou poucos concentram quase tudo.

**Técnica** histograma do grau e assortatividade de grau.

**Configuração** rede global sem peso. Assortatividade pelo coeficiente de Pearson do grau nas pontas das arestas.

**Resultados**

In [ ]:
rel = Relatorio("analise10", "ANÁLISE 10. Distribuição dos graus")

contagem = Counter(d for _, d in G.degree())
rel.escrever("personagens por grau (grau: quantidade):")
rel.escrever(", ".join(f"{g}: {n}" for g, n in sorted(contagem.items())))
rel.escrever(f"assortatividade de grau: {nx.degree_assortativity_coefficient(G):.3f}")
rel.salvar()

grau_david = G.degree("DavidMartinez")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(contagem.keys()), list(contagem.values()), color=COR_BARRA, width=0.7)
ax.set_xlabel("grau (número de personagens diferentes com quem interage)")
ax.set_ylabel("número de personagens")
ax.set_xticks(sorted(contagem))
ax.annotate("David", (grau_david, contagem[grau_david]), textcoords="offset points", xytext=(0, 6),
            ha="center", fontsize=8)
ax.set_title("Distribuição dos graus")
ax.grid(axis="x", visible=False)
salvar(fig, "fig10_distribuicao_dos_graus")

**Interpretação** 17 dos 37 personagens têm grau 1 ou 2, e no outro extremo o David tem grau 30, bem longe do segundo, que tem 13. É uma distribuição muito desigual, com muitos personagens de passagem e um centro forte. A assortatividade deu -0,44, então personagens com muitas conexões tendem a se ligar a personagens com poucas, o que bate com o David falando com vários personagens de um episódio só.

**Conclusões**
- Muitos personagens com poucas ligações e um centro muito forte.
- Assortatividade negativa, puxada pelo David.

### Análise 11. Equilíbrio estrutural da rede com sinal
**Pergunta e objetivo** os triângulos de personagens fecham de forma coerente, no estilo o inimigo do meu inimigo é meu amigo.

**Técnica** teoria do equilíbrio estrutural de Heider, formalizada por Cartwright e Harary. Um triângulo é equilibrado quando o produto dos três sinais é positivo. Técnica não listada no enunciado.

**Configuração** rede com o sinal líquido de cada par. Os pares neutros ficam fora, porque não têm sinal definido.

**Resultados**

In [ ]:
rel = Relatorio("analise11", "ANÁLISE 11. Equilíbrio estrutural da rede com sinal")

# rede só com os pares de sinal líquido definido; par neutro (soma zero) fica fora
liquido = sinal_por_par(df)
G_sinal = nx.Graph()
for (a, b), s in liquido.items():
    if s != "0":
        G_sinal.add_edge(a, b, sinal=1 if s == "+" else -1)
n_pos = sum(1 for *_, s in G_sinal.edges(data="sinal") if s > 0)
rel.escrever(f"pares com sinal líquido definido: {G_sinal.number_of_edges()} "
             f"({n_pos} positivos, {G_sinal.number_of_edges() - n_pos} negativos)")
rel.escrever(f"pares neutros, fora da análise: {sum(1 for s in liquido.values() if s == '0')}")

# triângulo equilibrado: produto dos três sinais positivo (Heider; Cartwright e Harary)
triangulos = [c for c in nx.enumerate_all_cliques(G_sinal) if len(c) == 3]


def produto(t):
    a, b, c = t
    return G_sinal[a][b]["sinal"] * G_sinal[b][c]["sinal"] * G_sinal[a][c]["sinal"]


equilibrados = [t for t in triangulos if produto(t) > 0]
rel.escrever(f"triângulos: {len(triangulos)}, equilibrados: {len(equilibrados)} "
             f"({len(equilibrados) / len(triangulos):.0%})")
rel.escrever("\ntriângulos desequilibrados:")
for t in triangulos:
    if produto(t) < 0:
        rel.escrever("  " + ", ".join(nome(v) for v in t))
rel.salvar()

fig, ax = plt.subplots(figsize=(13, 10))
desenhar_rede(ax, G_sinal, sinais={k: s for k, s in liquido.items() if s != "0"})
legenda_grupos(fig, G_sinal)
legenda_sinais(fig, sinais=("+", "-"))
ax.set_title("Rede com o sinal líquido de cada par (pares neutros fora)")
salvar(fig, "fig11_rede_sinal_liquido")

**Interpretação** Dos 84 pares, 73 têm sinal líquido definido, 45 positivos e 28 negativos. Eles formam 45 triângulos, e 37 deles (82%) são equilibrados. Na maior parte da rede vale a lógica de grupo, em que o inimigo comum une a equipe. Dos 8 triângulos desequilibrados, três têm o Falco e o Maine, porque o Falco enfrenta o Maine durante a cyberpsicose. Os outros vêm da minha regra de sinal. Como o sinal segue o resultado para a equipe, o David vencendo a Militech entra como `+`, e o triângulo com quem foi atacado pela Militech fica desequilibrado. A teoria pensa o sinal como afeto e na minha base ele é resultado.

**Conclusões**
- 82% dos triângulos são equilibrados.
- O Falco contra o Maine em surto é a principal tensão dentro da equipe.
- Parte do desequilíbrio vem da regra de sinal por resultado, o que é uma limitação para aplicar a teoria.

### Análise 12. A absorção do David pela equipe e a troca de liderança
**Pergunta e objetivo** que parte das interações do David acontece com a equipe em cada episódio, e o que muda antes e depois da morte do Maine.

**Técnica** proporção das interações do David com membros da equipe por episódio, e comparação de grau ponderado e intermediação entre os episódios 1 a 5 e 6 a 10. Técnica não listada no enunciado.

**Configuração** equipe definida como Maine, Lucy, Kiwi, Dorio, Pilar, Rebecca, Falco e Julio. A Kiwi conta como equipe mesmo traindo no fim. O corte no episódio 6 é a morte do Maine.

**Resultados**

In [ ]:
rel = Relatorio("analise12", "ANÁLISE 12. A absorção do David pela equipe e a troca de liderança")

# que parte das interações do David, em cada episódio, é com a equipe de mercenários
linhas = []
for ep in EPISODIOS:
    do_ep = df[df["episodio"] == ep]
    do_david = do_ep[(do_ep["personagem1"] == "DavidMartinez") | (do_ep["personagem2"] == "DavidMartinez")]
    outro = do_david["personagem1"].where(do_david["personagem1"] != "DavidMartinez", do_david["personagem2"])
    linhas.append({"episodio": ep, "interacoes_David": len(do_david), "com_equipe": int(outro.isin(EQUIPE).sum())})
absorcao = pd.DataFrame(linhas).set_index("episodio")
absorcao["proporcao_equipe"] = (absorcao["com_equipe"] / absorcao["interacoes_David"]).round(2)
rel.tabela(absorcao)

# antes e depois da morte do Maine (episódio 6)
rel.escrever("")
for rotulo, parte in {"episódios 1 a 5": df[df["episodio"] <= "E05"], "episódios 6 a 10": df[df["episodio"] >= "E06"]}.items():
    h = montar_grafo(parte)
    inter = pd.Series(nx.betweenness_centrality(h)).sort_values(ascending=False).round(2)
    cenas = pd.Series(dict(h.degree(weight="weight"))).sort_values(ascending=False)
    rel.escrever(f"{rotulo}: {h.number_of_nodes()} personagens, David em {cenas['DavidMartinez']} de {len(parte)} registros")
    rel.escrever("  maiores graus ponderados: " + ", ".join(f"{nome(k)} {v}" for k, v in cenas.head(5).items()))
    rel.escrever("  maiores intermediações:   " + ", ".join(f"{nome(k)} {v}" for k, v in inter.head(5).items()))
rel.salvar()

rotulos = [f"E{int(e[1:])}" for e in absorcao.index]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(rotulos, absorcao["proporcao_equipe"] * 100, color=COR_BARRA, width=0.62)
for x, v in zip(rotulos, absorcao["proporcao_equipe"]):
    ax.text(x, v * 100 + 2, f"{v:.0%}", ha="center", fontsize=8, color="#52514e")
ax.axvline(5.42, color="#52514e", linestyle="--", lw=1)
ax.text(5.48, 104, "morte do Maine", fontsize=8, color="#52514e")
ax.set_ylim(0, 112)
ax.set_ylabel("% das interações do David com a equipe")
ax.set_title("Interações do David com a equipe de mercenários, por episódio")
ax.grid(axis="x", visible=False)
salvar(fig, "fig12_absorcao_do_david")

**Interpretação** No episódio 1 nenhuma interação do David é com a equipe. No episódio 2 já são 56% e no episódio 3, quando o Maine aceita ele, 77%, com pico de 90% no episódio 6. A queda para 33% no episódio 10 é o David sozinho contra a Arasaka e o Adam Smasher. Comparando as duas metades, na primeira o Maine é o segundo em cenas e em intermediação. Na segunda a intermediação do David sobe de 0,49 para 0,69, e o segundo nível em cenas passa a ser Rebecca, Lucy e Falco.

**Conclusões**
- O David passa de 0% de interações com a equipe no episódio 1 para 77% no episódio 3.
- Depois da morte do Maine, a intermediação do David sobe de 0,49 para 0,69, o que bate com ele assumindo a liderança.

### Análise 13. O peso de cada perda na rede
**Pergunta e objetivo** a cada morte da história, o que se quebra na rede e quem perde a ligação com o resto. É a análise principal do trabalho.

**Técnica** remoção de vértice no momento da morte. Monto a rede acumulada até a cena da morte, retiro o personagem e comparo os componentes. Junto mostro a intermediação que ele tinha naquele momento. Técnica não listada no enunciado.

**Configuração** mortes com episódio e minuto da cena: Glória (E01 18:54), Pilar (E04 15:52), Dorio (E06 18:31), Maine (E06 22:49), Rebecca (E10 20:21) e David (E10 23:31). A morte da Kiwi não está nas cenas anotadas, então ficou de fora.

**Resultados**

In [ ]:
rel = Relatorio("analise13", "ANÁLISE 13. O peso de cada perda na rede")

# personagem, episódio e minuto da cena da morte
MORTES = [("GloriaMartinez", "E01", "18:54"), ("Pilar", "E04", "15:52"), ("Dorio", "E06", "18:31"),
          ("Maine", "E06", "22:49"), ("Rebecca", "E10", "20:21"), ("DavidMartinez", "E10", "23:31")]


def rede_ate(ep, minuto):
    """Rede acumulada até a cena indicada."""
    return montar_grafo(df[(df["episodio"] < ep) | ((df["episodio"] == ep) & (df["minuto"] <= minuto))])


# para cada morte: tiro o personagem da rede daquele momento e vejo o que se quebra
linhas = []
for morto, ep, minuto in MORTES:
    antes = rede_ate(ep, minuto)
    depois = antes.copy()
    depois.remove_node(morto)
    comps_antes = list(nx.connected_components(antes))
    comps_depois = list(nx.connected_components(depois))
    principal_antes = max(comps_antes, key=len)
    principal_depois = max(comps_depois, key=len)
    perderam = sorted(nome(v) for v in principal_antes - principal_depois - {morto})
    linhas.append({
        "personagem": nome(morto), "cena": f"{ep} {minuto}",
        "intermediacao": round(nx.betweenness_centrality(antes)[morto], 2),
        "componentes": f"{len(comps_antes)} -> {len(comps_depois)}",
        "maior_componente": f"{len(principal_antes)} -> {len(principal_depois)}",
        "perdem a ligação": ", ".join(perderam) if perderam else "-",
    })
impacto = pd.DataFrame(linhas).set_index("personagem")
rel.tabela(impacto)

# interações só entre membros da equipe, por episódio e sinal
EQUIPE_COMPLETA = EQUIPE | {"DavidMartinez"}
da_equipe = df[df["personagem1"].isin(EQUIPE_COMPLETA) & df["personagem2"].isin(EQUIPE_COMPLETA)]
por_ep = da_equipe.groupby(["episodio", "sinal"]).size().unstack(fill_value=0).reindex(
    index=EPISODIOS, columns=["+", "-", "0"], fill_value=0)
rel.escrever("\ninterações dentro da equipe por episódio e sinal:")
rel.tabela(por_ep)
rel.salvar()

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(impacto.index, impacto["intermediacao"], color=COR_BARRA, width=0.6)
for x, v in enumerate(impacto["intermediacao"]):
    ax.text(x, v + 0.015, f"{v:.2f}", ha="center", fontsize=8, color="#52514e")
ax.set_ylabel("intermediação no momento da morte")
ax.set_title("Intermediação de cada personagem na cena da sua morte")
ax.grid(axis="x", visible=False)
salvar(fig, "fig13a_intermediacao_na_morte")

fig, ax = plt.subplots(figsize=(9, 4.4))
base = None
for s in ["+", "-", "0"]:
    ax.bar(range(len(EPISODIOS)), por_ep[s], bottom=base, color=COR_SINAL[s], label=NOME_SINAL[s], width=0.62,
           edgecolor="white", linewidth=1.5)
    base = por_ep[s] if base is None else base + por_ep[s]
topo = base.max()
for morto, ep, minuto in MORTES:
    x = EPISODIOS.index(ep) + int(minuto[:2]) / 25 - 0.5
    ax.axvline(x, color="#52514e", linestyle="--", lw=0.8)
    ax.text(x + 0.04, topo * 1.12, nome(morto), rotation=90, fontsize=7.5, va="top")
ax.set_ylim(0, topo * 1.18)
ax.set_xticks(range(len(EPISODIOS)))
ax.set_xticklabels([f"E{int(e[1:])}" for e in EPISODIOS])
ax.set_ylabel("interações entre membros da equipe")
ax.set_title("Interações dentro da equipe por episódio, com as mortes marcadas")
ax.legend(title="Sinal", loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8, title_fontsize=8)
ax.grid(axis="x", visible=False)
salvar(fig, "fig13b_equipe_e_mortes")

# a rede no momento da morte do Maine, e a mesma rede sem ele
antes = rede_ate("E06", "22:49")
depois = antes.copy()
depois.remove_node("Maine")
principal = max(nx.connected_components(depois), key=len)
COR_ESTADO = {"Maine": "#f08c00", "continua ligado": "#2a78d6", "perde a ligação": "#d62f2f"}

fig, ax = plt.subplots(figsize=(13, 10))
desenhar_rede(ax, antes, cor_vertice={v: COR_ESTADO["Maine" if v == "Maine" else "continua ligado"] for v in antes})
fig.legend(handles=[Patch(color=COR_ESTADO[k], label=k) for k in ("Maine", "continua ligado")],
           loc="lower left", bbox_to_anchor=(0.06, 0.0), fontsize=9, ncol=2)
ax.set_title("Rede acumulada até a morte do Maine (episódio 6, 22:49)")
salvar(fig, "fig13c_rede_antes_da_morte_do_maine")

fig, ax = plt.subplots(figsize=(13, 10))
desenhar_rede(ax, depois, cor_vertice={v: COR_ESTADO["continua ligado" if v in principal else "perde a ligação"]
                                      for v in depois})
fig.legend(handles=[Patch(color=COR_ESTADO[k], label=k) for k in ("continua ligado", "perde a ligação")],
           loc="lower left", bbox_to_anchor=(0.06, 0.0), fontsize=9, ncol=2)
ax.set_title("A mesma rede sem o Maine: quem perde a ligação com o grupo principal")
salvar(fig, "fig13d_rede_depois_da_morte_do_maine")

**Interpretação** As perdas não pesam igual, e o peso de cada uma não depende de quanto o personagem aparece. A Glória aparece pouco, mas quando morre tem intermediação de 0,24, e sem ela o Maine e o diretor da Academia perdem a ligação com o resto. Ela era a única ponte entre a vida de estudante do David e os mercenários.

A morte do Maine é a que mais quebra a equipe. Ele tinha intermediação de 0,40 e sem ele a rede passa de 1 para 4 componentes. Falco e Faraday perdem a ligação, porque até ali só se ligavam à equipe pelo Maine. Depois disso o David passa a tratar direto com o Faraday, e é nessa relação que vem a traição.

Pilar, Dorio e Rebecca saem sem quebrar a estrutura, porque estavam no meio de um grupo bem ligado. A morte do David é a maior de todas, com 6 personagens soltos. No gráfico da equipe, o auge é o episódio 5, sem nenhuma interação negativa, e a quebra é o episódio 6, com 11 negativas.

**Conclusões**
- As mortes da Glória e do Maine são as que mais mudam a rede, porque os dois eram pontes.
- Sem o Maine, Falco e Faraday perdem a ligação com a equipe.
- Pilar, Dorio e Rebecca saem sem quebrar a estrutura.

## 9. Síntese dos resultados
**Pergunta 1, quem sustenta a rede.** O David é o centro, com intermediação de 0,67, e sem ele 6 personagens ficam isolados. As mortes que mais mudam a rede são a da Glória e a do Maine, porque os dois eram pontes no momento em que morrem. Pilar, Dorio e Rebecca saem sem quebrar nada.

**Pergunta 2, como a equipe se forma e se desfaz.** No episódio 1 nenhuma interação do David é com os mercenários e no episódio 3 já são 77%. A equipe chega ao auge no episódio 5 e quebra no episódio 6, com a cyberpsicose do Maine. Depois disso o David ocupa o lugar do Maine, com a intermediação subindo de 0,49 para 0,69.

**Pergunta 3, se as traições aparecem na rede.** Aparecem de dois jeitos. A da Kiwi é uma virada de sinal com a Lucy e com o Falco. A do Faraday não é virada, porque ele nunca colabora com a equipe, e as comunidades colocam ele junto da Arasaka e fora da equipe em todas as execuções.

**Limitações**
- A rede é pequena, com 37 personagens, então modularidade e assortatividade mudam bastante com poucos registros.
- O sinal segue o resultado para a equipe do David, então a rede conta a história do ponto de vista dele.
- O peso conta cenas, e não a duração ou a importância de cada uma.
- A rede não é direcionada, então não mostra quem age sobre quem.
- A coleta foi feita por uma pessoa só, sem uma segunda conferência independente.
- Grupos sem nome entram como um personagem só (tiras, Militech, Trauma Team, inimigos), o que reduz o tamanho real dos confrontos.
- A base só tem cenas com minuto anotado, então relações que não viram cena, como parentesco, não aparecem.

## 10. Referências
- Apoorv Agarwal, Augusto Corvalan, Jacob Jensen, and Owen Rambow. Social network analysis of Alice in Wonderland. In Proceedings of the NAACL-HLT 2012 Workshop on Computational Linguistics for Literature, pages 88-96. Association for Computational Linguistics, 2012. https://aclanthology.org/W12-2513/
- Apoorv Agarwal, Sriramkumar Balasubramanian, Jiehan Zheng, and Sarthak Dash. Parsing screenplays for extracting social networks from movies. In Proceedings of the 3rd Workshop on Computational Linguistics for Literature, pages 50-58. Association for Computational Linguistics, 2014. doi: 10.3115/v1/W14-0907.
- Vincent Labatut and Xavier Bost. Extraction and analysis of fictional character networks: A survey. ACM Computing Surveys, 52(5):1-40, 2019. doi: 10.1145/3344548.
- Cyberpunk Wiki (Fandom). Páginas de personagens e episódios de Cyberpunk: Edgerunners. https://cyberpunk.fandom.com/ Consulta em 15/11/2026.